# Heart Disease Dataset Preparation

This notebook combines the four UCI Heart Disease dataset files—Cleveland, Hungarian, Switzerland, and VA—into one consistent dataset. The resulting `data/raw/heart.csv` is intended to provide a single, reproducible input for downstream exploratory data analysis (EDA), model training, MLflow experiment tracking, FastAPI model serving, Docker packaging, CI/CD, and Kubernetes deployment.

The workflow preserves all source observations, records missing values explicitly, standardizes the target for binary classification, and performs basic validation before saving the consolidated CSV. Run the notebook from the repository root or from the `notebooks/` directory.

## 1. Imports and project configuration

The project root is discovered from the notebook's current working directory, so the same relative data paths work when Jupyter is launched from either the repository root or the `notebooks/` directory. The expected column schema is defined once and reused for every source file.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

candidate_roots = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    (candidate for candidate in candidate_roots if (candidate / "data" / "raw").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find the project data/raw directory from the current working "
        f"directory: {Path.cwd()}"
    )

raw_data_dir = project_root / "data" / "raw"
columns = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal",
    "target",
]

print(f"Project root: {project_root}")
print(f"Raw data directory: {raw_data_dir}")
print(f"Expected number of columns: {len(columns)}")

Project root: c:\Users\kshvj\OneDrive\Documents\LJ's creations\MTECH\Git projects\heart-disease-pipeline-mlops-bits-assignment
Raw data directory: c:\Users\kshvj\OneDrive\Documents\LJ's creations\MTECH\Git projects\heart-disease-pipeline-mlops-bits-assignment\data\raw
Expected number of columns: 14


## 2. Load the four source datasets

Each source file contains comma-separated records without a header row. The `?` marker denotes an unknown value and is parsed as a missing value (`NaN`). Every file is loaded separately with the shared column names. Missing or unreadable/malformed inputs raise an explicit error rather than being silently omitted; each successful load reports its row count.

In [2]:
source_files = {
    "Cleveland": "processed.cleveland.data",
    "Hungarian": "processed.hungarian.data",
    "Switzerland": "processed.switzerland.data",
    "VA": "processed.va.data",
}

datasets = {}
for dataset_name, filename in source_files.items():
    source_path = raw_data_dir / filename
    if not source_path.is_file():
        raise FileNotFoundError(f"Required {dataset_name} source file not found: {source_path}")

    try:
        datasets[dataset_name] = pd.read_csv(
            source_path,
            names=columns,
            na_values="?",
            header=None,
        )
    except Exception as error:
        raise RuntimeError(
            f"Could not read {dataset_name} source file at {source_path}: {error}"
        ) from error

    print(f"{dataset_name}: {len(datasets[dataset_name]):,} rows loaded from {filename}")

Cleveland: 303 rows loaded from processed.cleveland.data
Hungarian: 294 rows loaded from processed.hungarian.data
Switzerland: 123 rows loaded from processed.switzerland.data
VA: 200 rows loaded from processed.va.data


## 3. Inspect each source dataset

Before combining the data, inspect the dimensions and first five records from every source. This provides a quick check that the files were parsed independently, that the expected schema was applied, and that the values appear in the expected columns.

In [3]:
for dataset_name, dataset in datasets.items():
    print(f"\n{dataset_name} dataset shape: {dataset.shape}")
    display(dataset.head(5))


Cleveland dataset shape: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0



Hungarian dataset shape: (294, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,28,1,2,130.0,132.0,0.0,2.0,185.0,0.0,0.0,NaN,NaN,NaN,0
1,29,1,2,120.0,243.0,0.0,0.0,160.0,0.0,0.0,NaN,NaN,NaN,0
2,29,1,2,140.0,NaN,0.0,0.0,170.0,0.0,0.0,NaN,NaN,NaN,0
3,30,0,1,170.0,237.0,0.0,1.0,170.0,0.0,0.0,NaN,NaN,6.0,0
4,31,0,2,100.0,219.0,0.0,1.0,150.0,0.0,0.0,NaN,NaN,NaN,0



Switzerland dataset shape: (123, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,32,1,1,95.0,0,NaN,0.0,127.0,0.0,0.7,1.0,NaN,NaN,1
1,34,1,4,115.0,0,NaN,NaN,154.0,0.0,0.2,1.0,NaN,NaN,1
2,35,1,4,NaN,0,NaN,0.0,130.0,1.0,NaN,NaN,NaN,7.0,3
3,36,1,4,110.0,0,NaN,0.0,125.0,1.0,1.0,2.0,NaN,6.0,1
4,38,0,4,105.0,0,NaN,0.0,166.0,0.0,2.8,1.0,NaN,NaN,2



VA dataset shape: (200, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,4,140.0,260.0,0.0,1,112.0,1.0,3.0,2.0,NaN,NaN,2
1,44,1,4,130.0,209.0,0.0,1,127.0,0.0,0.0,NaN,NaN,NaN,0
2,60,1,4,132.0,218.0,0.0,1,140.0,1.0,1.5,3.0,NaN,NaN,2
3,55,1,4,142.0,228.0,0.0,1,149.0,1.0,2.5,1.0,NaN,NaN,1
4,66,1,3,110.0,213.0,1.0,2,99.0,1.0,1.3,2.0,NaN,NaN,0


## 4. Combine the source datasets

The four dataframes share the same column names and order, so they can be concatenated row-wise. A fresh integer index is created for the unified dataset. The following outputs report its size, show a preview, summarize column data types, and quantify missing values so that later preprocessing decisions are visible.

In [4]:
heart_df = pd.concat(datasets.values(), ignore_index=True)

print(f"Total rows: {len(heart_df):,}")
print(f"Total columns: {heart_df.shape[1]:,}")
display(heart_df.head(10))

print("\nDataframe information:")
heart_df.info()

print("\nMissing values by column:")
display(heart_df.isna().sum().rename("missing_values").to_frame())

Total rows: 920
Total columns: 14


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0
5,56.0,1.0,2.0,120.0,236.0,0.0,0.0,178.0,0.0,0.8,1.0,0.0,3.0,0
6,62.0,0.0,4.0,140.0,268.0,0.0,2.0,160.0,0.0,3.6,3.0,2.0,3.0,3
7,57.0,0.0,4.0,120.0,354.0,0.0,0.0,163.0,1.0,0.6,1.0,0.0,3.0,0
8,63.0,1.0,4.0,130.0,254.0,0.0,2.0,147.0,0.0,1.4,2.0,1.0,7.0,2
9,53.0,1.0,4.0,140.0,203.0,1.0,2.0,155.0,1.0,3.1,3.0,0.0,7.0,1



Dataframe information:
<class 'pandas.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       920 non-null    float64
 1   sex       920 non-null    float64
 2   cp        920 non-null    float64
 3   trestbps  861 non-null    float64
 4   chol      890 non-null    float64
 5   fbs       830 non-null    float64
 6   restecg   918 non-null    float64
 7   thalach   865 non-null    float64
 8   exang     865 non-null    float64
 9   oldpeak   858 non-null    float64
 10  slope     611 non-null    float64
 11  ca        309 non-null    float64
 12  thal      434 non-null    float64
 13  target    920 non-null    int64  
dtypes: float64(13), int64(1)
memory usage: 100.8 KB

Missing values by column:


,missing_values
age,0
sex,0
cp,0
trestbps,59
chol,30
fbs,90
restecg,2
thalach,55
exang,55
oldpeak,62


## 5. Understand and standardize the target

In the original UCI data, `target = 0` means no heart disease, while target values `1`, `2`, `3`, and `4` indicate the presence of heart disease at different levels. For this project's initial prediction workflow, a binary target is more suitable: it directly represents the distinction between absence and presence of disease and supports a straightforward binary-classification task.

The original class distribution is displayed first. Before conversion, target values are checked for missing, non-numeric, or unexpected values; invalid targets stop processing with a clear error rather than being silently assigned a class.

In [5]:
print("Original target class distribution:")
display(heart_df["target"].value_counts(dropna=False).sort_index().rename("count").to_frame())

target_numeric = pd.to_numeric(heart_df["target"], errors="coerce")
non_numeric_target = heart_df["target"].notna() & target_numeric.isna()
unexpected_target_values = sorted(
    set(target_numeric.dropna().unique()) - {0, 1, 2, 3, 4}
)

if heart_df["target"].isna().any():
    raise ValueError("Target contains missing values; resolve them before binary conversion.")
if non_numeric_target.any():
    invalid_values = heart_df.loc[non_numeric_target, "target"].unique().tolist()
    raise ValueError(f"Target contains non-numeric values: {invalid_values}")
if unexpected_target_values:
    raise ValueError(f"Target contains unexpected class values: {unexpected_target_values}")

heart_df["target"] = np.where(target_numeric.eq(0), 0, 1).astype("int64")

print("Binary target class distribution:")
display(heart_df["target"].value_counts().sort_index().rename("count").to_frame())

Original target class distribution:


,count
target,
0,411
1,265
2,109
3,107
4,28


Binary target class distribution:


,count
target,
0,411
1,509


## 6. Basic data validation

Run basic checks on the consolidated dataframe before writing it to disk. Duplicate records are counted but retained so that this preparation step does not discard source observations without an explicit project decision. Missing-value totals, column types, and the transformed target distribution are also reported for review.

In [6]:
duplicate_count = int(heart_df.duplicated().sum())
missing_value_count = int(heart_df.isna().sum().sum())

print(f"Duplicate row count (retained): {duplicate_count:,}")
print(f"Total missing value count: {missing_value_count:,}")

print("\nData types:")
display(heart_df.dtypes.rename("dtype").to_frame())

print("\nBinary target class distribution:")
display(heart_df["target"].value_counts().sort_index().rename("count").to_frame())

Duplicate row count (retained): 2
Total missing value count: 1,759

Data types:


,dtype
age,float64
sex,float64
cp,float64
trestbps,float64
chol,float64
fbs,float64
restecg,float64
thalach,float64
exang,float64
oldpeak,float64



Binary target class distribution:


,count
target,
0,411
1,509


## 7. Save and verify the prepared dataset

Save the unified dataframe to `data/raw/heart.csv` without writing the dataframe index as an extra feature. The file is then reloaded from disk, and its shape and first five rows are displayed to verify that the expected artifact was created and can be read back.

In [7]:
output_path = raw_data_dir / "heart.csv"
heart_df.to_csv(output_path, index=False)
print(f"Prepared dataset saved to: {output_path}")

saved_heart_df = pd.read_csv(output_path)
print(f"Reloaded dataset shape: {saved_heart_df.shape}")
display(saved_heart_df.head(5))

Prepared dataset saved to: c:\Users\kshvj\OneDrive\Documents\LJ's creations\MTECH\Git projects\heart-disease-pipeline-mlops-bits-assignment\data\raw\heart.csv
Reloaded dataset shape: (920, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


## Summary

Four UCI Heart Disease datasets—Cleveland, Hungarian, Switzerland, and VA—were combined into a unified dataset with **920 records**. The prepared data has **13 predictor features** and **one binary target** (14 columns in total). All source observations are retained, and the target now consistently represents no disease (`0`) versus disease present (`1`).

This consolidated schema and saved CSV provide a clear, reusable starting point for EDA and machine-learning workflows. The validation outputs document remaining missing values and duplicates; downstream analysis and training should select an appropriate missing-data strategy, check modeling assumptions, and use suitable evaluation methods before deployment.